# Lecture 25 — Offline RL: IQL, CQL, and Game Theory (Nash Equilibria)

This notebook implements the core algorithms and demos from Lecture 25:

- **Implicit Q-Learning (IQL)**: expectile regression for V, TD Q-learning, and AWR policy update
- **Conservative Q-Learning (CQL)**: conservative regularizer added to Bellman error
- **Game Theory**: normal-form games solved via Nash equilibrium (nashpy)

Each section includes small demo cells and lightweight unit tests to validate shapes and key properties.

In [ ]:
# Setup & dependencies
import os
import random
import math
from typing import Tuple, Dict, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import seaborn as sns

# optional: nashpy used for game-theory examples
try:
    import nashpy as nash
except Exception:
    nash = None

# Device
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

# Reproducibility note
print("Note: For reproducible runs, set seeds using the helper in the utilities cell.")

In [ ]:
# Utility helpers: seeding, device helpers, simple logging, and a lightweight ReplayBuffer

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

class SimpleLogger:
    def __init__(self):
        self._data = {}
    def add(self, key, value):
        self._data.setdefault(key, []).append(value)
    def mean(self, key):
        v = self._data.get(key, [])
        return float(np.mean(v)) if v else float('nan')
    def clear(self):
        self._data.clear()

class ReplayBuffer:
    def __init__(self, state_dim, action_dim, capacity=100000):
        self.capacity = int(capacity)
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.ptr = 0
        self.size = 0
        self.states = np.zeros((self.capacity, state_dim), dtype=np.float32)
        self.actions = np.zeros((self.capacity, action_dim), dtype=np.float32)
        self.rewards = np.zeros((self.capacity, 1), dtype=np.float32)
        self.next_states = np.zeros((self.capacity, state_dim), dtype=np.float32)
        self.dones = np.zeros((self.capacity, 1), dtype=np.float32)

    def add(self, s, a, r, s2, d):
        self.states[self.ptr] = s
        self.actions[self.ptr] = a
        self.rewards[self.ptr] = r
        self.next_states[self.ptr] = s2
        self.dones[self.ptr] = d
        self.ptr = (self.ptr + 1) % self.capacity
        self.size = min(self.size + 1, self.capacity)

    def sample(self, batch_size):
        idx = np.random.randint(0, self.size, size=batch_size)
        return (
            torch.from_numpy(self.states[idx]).to(DEVICE),
            torch.from_numpy(self.actions[idx]).to(DEVICE),
            torch.from_numpy(self.rewards[idx]).to(DEVICE),
            torch.from_numpy(self.next_states[idx]).to(DEVICE),
            torch.from_numpy(self.dones[idx]).to(DEVICE),
        )

# Quick smoke test for replay buffer
rb = ReplayBuffer(state_dim=3, action_dim=2, capacity=100)
for _ in range(50):
    rb.add(np.random.randn(3), np.random.randn(2), np.random.randn(1), np.random.randn(3), np.random.randint(0,2,1))
print("ReplayBuffer sample shapes:", [x.shape for x in rb.sample(8)])

In [ ]:
# Common NN modules and weight init

def build_mlp(input_dim, output_dim, hidden_dim=256, activation=nn.ReLU, output_activation=None):
    layers = [nn.Linear(input_dim, hidden_dim), activation()]
    layers += [nn.Linear(hidden_dim, hidden_dim), activation()]
    layers += [nn.Linear(hidden_dim, output_dim)]
    if output_activation is not None:
        layers.append(output_activation())
    return nn.Sequential(*layers)


def init_weights(m):
    if isinstance(m, nn.Linear):
        nn.init.xavier_uniform_(m.weight)
        if m.bias is not None:
            nn.init.constant_(m.bias, 0)

print('Helper modules ready')

In [ ]:
# IQL implementation (agent + helpers)
class IQLAgent(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256, expectile=0.7, discount=0.99, temperature=3.0, lr=3e-4, device=DEVICE):
        super().__init__()
        self.device = device
        self.expectile = float(expectile)
        self.gamma = float(discount)
        self.temperature = float(temperature)
        self.state_dim = state_dim
        self.action_dim = action_dim

        # Q networks
        self.q1_net = build_mlp(state_dim + action_dim, 1, hidden_dim).to(self.device)
        self.q2_net = build_mlp(state_dim + action_dim, 1, hidden_dim).to(self.device)
        self.q1_net.apply(init_weights)
        self.q2_net.apply(init_weights)

        # q_target is a copy of q1 for evaluation and stable q-value computation
        self.q_target = copy.deepcopy(self.q1_net).to(self.device)

        # V network
        self.v_net = build_mlp(state_dim, 1, hidden_dim).to(self.device)
        self.v_net.apply(init_weights)

        # Actor (outputs action mean)
        self.actor = build_mlp(state_dim, action_dim, hidden_dim).to(self.device)
        self.actor.apply(init_weights)

        # Optimizers
        self.v_optimizer = torch.optim.Adam(self.v_net.parameters(), lr=lr)
        self.q_optimizer = torch.optim.Adam(list(self.q1_net.parameters()) + list(self.q2_net.parameters()), lr=lr)
        self.actor_optimizer = torch.optim.Adam(self.actor.parameters(), lr=lr)

        # Polyak target update coeff
        self.tau = 0.005

    def expectile_loss(self, diff: torch.Tensor):
        # diff: (q - v)
        weight = torch.where(diff > 0, self.expectile, 1 - self.expectile)
        return torch.mean(weight * (diff ** 2))

    def sample_action(self, state: torch.Tensor, deterministic=False):
        mean = self.actor(state)
        if deterministic:
            return torch.tanh(mean)
        # simple gaussian exploration with small std
        std = 0.1
        noise = torch.randn_like(mean) * std
        return torch.tanh(mean + noise)

    def update_target(self):
        for p, t in zip(self.q1_net.parameters(), self.q_target.parameters()):
            t.data.copy_(self.tau * p.data + (1 - self.tau) * t.data)

    def update(self, states, actions, rewards, next_states, dones):
        # ensure shapes are correct and tensors are floats
        states = states.float().to(self.device)
        actions = actions.float().to(self.device)
        rewards = rewards.float().to(self.device)
        next_states = next_states.float().to(self.device)
        dones = dones.float().to(self.device)

        # --- Step 1: Update V via expectile regression ---
        with torch.no_grad():
            q1 = self.q_target(torch.cat([states, actions], dim=1))
            q2 = self.q2_net(torch.cat([states, actions], dim=1))
            q_target = torch.min(q1, q2)

        v_pred = self.v_net(states)
        v_loss = self.expectile_loss(q_target - v_pred)
        self.v_optimizer.zero_grad()
        v_loss.backward()
        self.v_optimizer.step()

        # --- Step 2: Update Q(s,a) using V(s') ---
        with torch.no_grad():
            next_v = self.v_net(next_states)
            q_target_val = rewards + self.gamma * (1 - dones) * next_v

        q1_pred = self.q1_net(torch.cat([states, actions], dim=1))
        q2_pred = self.q2_net(torch.cat([states, actions], dim=1))
        q_loss = F.mse_loss(q1_pred, q_target_val) + F.mse_loss(q2_pred, q_target_val)

        self.q_optimizer.zero_grad()
        q_loss.backward()
        self.q_optimizer.step()

        # --- Step 3: Actor via AWR (weighted BC) ---
        with torch.no_grad():
            q_val = self.q_target(torch.cat([states, actions], dim=1))
            v_val = self.v_net(states)
            advantage = q_val - v_val
            # compute weights: exp(adv / temp)
            exp_adv = torch.exp((advantage / (self.temperature + 1e-8)))
            exp_adv = torch.clamp(exp_adv, max=100.0)
            exp_adv = exp_adv.detach()

        pred_actions = self.actor(states)
        actor_loss = torch.mean(exp_adv * ((pred_actions - actions) ** 2))

        self.actor_optimizer.zero_grad()
        actor_loss.backward()
        self.actor_optimizer.step()

        # update q target network slowly
        self.update_target()

        return v_loss.item(), q_loss.item(), actor_loss.item()

# Small demo to validate shapes and losses
set_seed(0)
agent = IQLAgent(state_dim=3, action_dim=2, hidden_dim=64, expectile=0.7, temperature=3.0)
# create a small random batch
batch = 16
states = torch.randn(batch, 3)
actions = torch.randn(batch, 2)
rewards = torch.randn(batch, 1)
next_states = torch.randn(batch, 3)
dones = torch.zeros(batch, 1)

v_l, q_l, a_l = agent.update(states, actions, rewards, next_states, dones)
print(f"IQL demo losses — V: {v_l:.4f}, Q: {q_l:.4f}, Actor: {a_l:.4f}")

In [ ]:
# IQL training loop (toy) + checkpointing + basic tests
set_seed(42)
agent = IQLAgent(state_dim=3, action_dim=2, hidden_dim=64)
# toy buffer
rb = ReplayBuffer(state_dim=3, action_dim=2, capacity=200)
for _ in range(200):
    rb.add(np.random.randn(3), np.random.randn(2), np.random.randn(1), np.random.randn(3), np.random.randint(0,2,1))

logger = SimpleLogger()
for it in range(10):
    s, a, r, s2, d = rb.sample(32)
    v_l, q_l, a_l = agent.update(s, a, r, s2, d)
    logger.add('v_loss', v_l); logger.add('q_loss', q_l); logger.add('actor_loss', a_l)

print('IQL: mean losses', logger.mean('v_loss'), logger.mean('q_loss'), logger.mean('actor_loss'))

# Checkpoint save/load
ckpt = {'q1': agent.q1_net.state_dict(), 'q2': agent.q2_net.state_dict(), 'v': agent.v_net.state_dict(), 'actor': agent.actor.state_dict()}
ckpt_path = 's25_iql_ckpt.pt'
torch.save(ckpt, ckpt_path)
loaded = torch.load(ckpt_path)
# quick sanity
assert 'q1' in loaded and 'actor' in loaded
print('Checkpoint saved and verified')

# Simple tests
# expectile loss behaviour
diff_pos = torch.tensor([1.0, 2.0])
diff_neg = torch.tensor([-1.0, -2.0])
L_pos = agent.expectile_loss(diff_pos)
L_neg = agent.expectile_loss(diff_neg)
assert L_pos >= 0 and L_neg >= 0
print('Basic expectile tests passed')

In [ ]:
# IQL: simple evaluation on a synthetic toy dynamics

def toy_step(state, action):
    # state, action: numpy arrays
    next_state = state + 0.1 * action
    reward = -np.linalg.norm(next_state)  # goal: stay near origin
    done = False
    return next_state, reward, done

# evaluate policy deterministically
set_seed(0)
agent = IQLAgent(state_dim=3, action_dim=2, hidden_dim=64)
# run a few episodes
returns = []
for ep in range(20):
    s = np.random.randn(3)
    total = 0.0
    for t in range(50):
        s_t = torch.from_numpy(s.astype(np.float32)).unsqueeze(0).to(DEVICE)
        a_t = agent.sample_action(s_t, deterministic=True).cpu().numpy()[0]
        s, r, d = toy_step(s, a_t)
        total += r
        if d:
            break
    returns.append(total)

plt.figure(figsize=(6,3)); plt.plot(returns, '-o'); plt.title('Toy eval returns (IQL deterministic)'); plt.xlabel('episode'); plt.ylabel('return'); plt.grid(True); plt.show()
print('Avg return:', np.mean(returns))

In [ ]:
# CQL implementation
class CQLAgent(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim=256, alpha=1.0, device=DEVICE):
        super().__init__()
        self.device = device
        self.action_dim = action_dim
        self.q_net = build_mlp(state_dim + action_dim, 1, hidden_dim).to(self.device)
        self.q_net.apply(init_weights)
        self.alpha = float(alpha)
        self.gamma = 0.99

    def get_policy_action(self, state: torch.Tensor):
        # For a minimal demo, return zeros of action shape
        batch = state.shape[0]
        return torch.zeros((batch, self.action_dim), dtype=torch.float32, device=state.device)

    def compute_cql_loss(self, states, actions, rewards, next_states, dones):
        states = states.float().to(self.device)
        actions = actions.float().to(self.device)
        rewards = rewards.float().to(self.device)
        next_states = next_states.float().to(self.device)
        dones = dones.float().to(self.device)

        # Bellman target (simple)
        with torch.no_grad():
            next_actions = self.get_policy_action(next_states)
            next_q = self.q_net(torch.cat([next_states, next_actions], dim=1))
            target_q = rewards + self.gamma * (1 - dones) * next_q

        current_q = self.q_net(torch.cat([states, actions], dim=1))
        bellman_loss = F.mse_loss(current_q, target_q)

        # Conservative regularizer: sample random actions uniformly in [-1,1]
        batch_size = states.shape[0]
        random_actions = (torch.rand((batch_size, self.action_dim), device=states.device) * 2.0 - 1.0)
        q_random = self.q_net(torch.cat([states, random_actions], dim=1))

        conservative = self.alpha * (torch.mean(q_random) - torch.mean(current_q))

        total = bellman_loss + conservative
        return total, bellman_loss.item(), conservative.item()

# Demo of CQL loss
set_seed(0)
cql = CQLAgent(state_dim=3, action_dim=2, alpha=1.0)
s = torch.randn(16,3)
a = torch.randn(16,2)
r = torch.randn(16,1)
s2 = torch.randn(16,3)
d = torch.zeros(16,1)
loss, bell, cons = cql.compute_cql_loss(s,a,r,s2,d)
print(f"CQL loss demo: total={loss.item():.4f}, bellman={bell:.4f}, conservative={cons:.4f}")

In [ ]:
# CQL training loop (toy) and conservative regularizer tests
set_seed(1)
# toy buffer
rb = ReplayBuffer(state_dim=3, action_dim=2, capacity=500)
for _ in range(200):
    rb.add(np.random.randn(3), np.random.randn(2), np.random.randn(1), np.random.randn(3), np.random.randint(0,2,1))

results = {}
for alpha in [0.0, 1.0, 5.0]:
    agent = CQLAgent(state_dim=3, action_dim=2, alpha=alpha)
    opt = torch.optim.Adam(agent.q_net.parameters(), lr=3e-4)
    logger = SimpleLogger()
    for it in range(20):
        s, a, r, s2, d = rb.sample(64)
        loss, bell, cons = agent.compute_cql_loss(s,a,r,s2,d)
        opt.zero_grad(); loss.backward(); opt.step()
        logger.add('total', loss.item()); logger.add('bell', bell); logger.add('cons', cons)
    results[alpha] = {'total': logger.mean('total'), 'bell': logger.mean('bell'), 'cons': logger.mean('cons')}

results

In [ ]:
# CQL evaluation: Q-value histograms (data vs random actions)
set_seed(2)
agent = CQLAgent(state_dim=3, action_dim=2, alpha=5.0)
# use some samples
s, a, r, s2, d = ReplayBuffer(3,2,capacity=100).sample(50)  # empty buffer => shapes ok (but content zeros)
# sample random actions
with torch.no_grad():
    q_data = agent.q_net(torch.cat([s, a], dim=1)).cpu().numpy().flatten()
    rand_a = (torch.rand((s.shape[0], agent.action_dim)) * 2.0 - 1.0)
    q_rand = agent.q_net(torch.cat([s, rand_a], dim=1)).cpu().numpy().flatten()

plt.figure(figsize=(6,3))
sns.histplot(q_data, color='C0', label='data actions', kde=False, stat='density', bins=20)
sns.histplot(q_rand, color='C1', label='random actions', kde=False, stat='density', bins=20)
plt.legend(); plt.title('Q-value distributions (data vs random)'); plt.show()

In [ ]:
# Game Theory: Nash equilibrium demonstration (nashpy)
if nash is None:
    print('nashpy not found. Install with: pip install nashpy')
else:
    import numpy as np
    # Prisoner's Dilemma
    A_payoff = np.array([[-1, -3], [0, -2]])
    B_payoff = np.array([[-1, 0], [-3, -2]])
    game = nash.Game(A_payoff, B_payoff)
    print('--- Prisoner\'s Dilemma ---')
    for eq in game.support_enumeration():
        a_strat, b_strat = eq
        print('A:', a_strat, 'B:', b_strat)

    # Matching pennies (zero-sum)
    A_mp = np.array([[1, -1], [-1, 1]])
    B_mp = -A_mp
    game2 = nash.Game(A_mp, B_mp)
    print('\n--- Matching Pennies ---')
    for eq in game2.support_enumeration():
        a_strat, b_strat = eq
        print('A:', a_strat, 'B:', b_strat)

    # Basic tests (where we expect pure defect/defect NE for PD)
    # Check if (Defect, Defect) is a Nash equilibrium by checking pure strategies
    def is_pure_ne(game, a_idx, b_idx):
        # check unilateral deviations
        uA = game[0][a_idx, b_idx]
        uB = game[1][a_idx, b_idx]
        # best responses
        a_best = np.argmax(game[0][:, b_idx])
        b_best = np.argmax(game[1][a_idx, :])
        return (a_best == a_idx) and (b_best == b_idx)

    if is_pure_ne(game, 1, 1):
        print('\nPrisoner\'s Dilemma: (Defect, Defect) is a pure NE as expected')

In [ ]:
# Notebook utilities: saving/loading models and reproducible checks

def save_agent(agent: nn.Module, path: str):
    torch.save(agent.state_dict(), path)

def load_agent(agent: nn.Module, path: str):
    agent.load_state_dict(torch.load(path, map_location=DEVICE))

# Reproducibility quick check
set_seed(123)
a1 = IQLAgent(3,2,hidden_dim=32)
out1 = a1.actor(torch.randn(4,3)).detach().cpu().numpy()
set_seed(123)
a2 = IQLAgent(3,2,hidden_dim=32)
out2 = a2.actor(torch.randn(4,3)).detach().cpu().numpy()
assert np.allclose(out1, out2), 'Reproducibility check failed'
print('Reproducibility smoke test passed')

print('To run tests in terminal: python -m pytest tests/test_s25.py -q')

# Notes & References

- IQL: Kumar et al., "Stabilizing Off-Policy Q-Learning via Bootstrapping" (lecture notes distill expectile + AWR steps)
- CQL: Kumar et al., "Conservative Q-Learning for Offline Reinforcement Learning"
- Game Theory: nashpy documentation and standard normal-form game definitions

Next steps / TODOs:
- Add a real environment (e.g., simple Gym env) and run full training/plots
- Add more comprehensive unit tests that execute relevant notebook cells
- Add hyperparameter search utilities and TensorBoard logging

---

End of `s25.ipynb`